# Breaking-Fake ViT Model Training in Google Colab

This notebook trains the Breaking-Fake Vision Transformer model on GPU with checkpoint resumption.

## Step 1: Mount Google Drive

Connect to your Google Drive to access your project files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")

## Step 2: Install Dependencies

Install required packages for training.

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install timm tqdm tensorboard -q
print("✓ All dependencies installed!")

# Verify installations
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Setup Project Path

Navigate to your Breaking-Fake project. Update the path if your folder structure is different.

In [ ]:
import os
import sys
from pathlib import Path

# TODO: Update this path to match your Google Drive structure
# Example: /content/drive/MyDrive/Breaking-Fake
PROJECT_PATH = '/content/drive/MyDrive/Breaking-Fake'

# Verify the path exists
if not os.path.exists(PROJECT_PATH):
    print(f"❌ Path not found: {PROJECT_PATH}")
    print("\nAvailable folders in your Drive:")
    !ls -la /content/drive/MyDrive/ | head -20
else:
    os.chdir(PROJECT_PATH)
    print(f"✓ Working directory: {os.getcwd()}")
    print(f"\n✓ Project structure:")
    !ls -la

## Step 4: Install Project Requirements

Install any additional dependencies from requirements.txt files.

In [ ]:
# Install model requirements
if os.path.exists('model/requirements.txt'):
    !pip install -r model/requirements.txt -q
    print("✓ Model requirements installed")
else:
    print("⚠ model/requirements.txt not found")

# Install backend requirements (if needed)
if os.path.exists('backend/requirements.txt'):
    !pip install -r backend/requirements.txt -q
    print("✓ Backend requirements installed")

print("\n✓ All requirements installed!")

## Step 5: Verify Data Structure

Check that your training data is available.

In [ ]:
# Check data structure
data_path = Path('model/data/raw')

print("Data structure:")
if data_path.exists():
    for item in data_path.iterdir():
        if item.is_dir():
            file_count = len(list(item.glob('*')))
            print(f"  📁 {item.name}: {file_count} files")
        else:
            print(f"  📄 {item.name}")
else:
    print(f"⚠ Data directory not found: {data_path}")

# Check artifacts and logs directories
artifacts_path = Path('model/artifacts')
logs_path = Path('model/logs')

artifacts_path.mkdir(parents=True, exist_ok=True)
logs_path.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Artifacts directory: {artifacts_path}")
print(f"✓ Logs directory: {logs_path}")

# List existing checkpoints
checkpoints = list(artifacts_path.glob('*.pth'))
if checkpoints:
    print(f"\nExisting checkpoints:")
    for ckpt in checkpoints:
        size_mb = ckpt.stat().st_size / (1024**2)
        print(f"  ✓ {ckpt.name} ({size_mb:.1f} MB)")
else:
    print("\n⚠ No checkpoints found (will train from scratch)")

## Step 6: Training Configuration

Configure training parameters.

In [ ]:
# Training configuration
CONFIG = {
    'data_dir': 'model/data/raw',
    'batch_size': 32,           # Adjust based on GPU memory
    'epochs': 20,               # Total number of epochs
    'learning_rate': 1e-4,
    'num_workers': 4,
    'device': 'cuda',           # Use GPU
    'resume': 'auto',           # Auto-resume from best checkpoint
}

print("Training Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Step 7: Run Training

Start the training process. This will automatically resume from the best checkpoint if one exists.

In [ ]:
import subprocess
import sys

# Build training command
cmd = [
    sys.executable, 'model/src/train.py',
    '--data-dir', CONFIG['data_dir'],
    '--batch-size', str(CONFIG['batch_size']),
    '--epochs', str(CONFIG['epochs']),
    '--lr', str(CONFIG['learning_rate']),
    '--device', CONFIG['device'],
    '--num-workers', str(CONFIG['num_workers']),
    '--resume', CONFIG['resume'],
]

print("Starting training...")
print(f"Command: {' '.join(cmd)}\n")

# Run training
result = subprocess.run(cmd, cwd=PROJECT_PATH)

if result.returncode == 0:
    print("\n✓ Training completed successfully!")
else:
    print(f"\n❌ Training failed with exit code {result.returncode}")

## Step 8: Check Training Results

Review the trained model and checkpoints.

In [ ]:
import os
from pathlib import Path

artifacts_dir = Path(PROJECT_PATH) / 'model' / 'artifacts'
logs_dir = Path(PROJECT_PATH) / 'model' / 'logs'

print("=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

# Check saved models
print("\n📁 Saved Models:")
if artifacts_dir.exists():
    model_files = sorted(artifacts_dir.glob('*.pth'))
    if model_files:
        for model_file in model_files:
            size_mb = model_file.stat().st_size / (1024**2)
            print(f"  ✓ {model_file.name} ({size_mb:.1f} MB)")
    else:
        print("  ⚠ No model files found")
else:
    print(f"  ⚠ Artifacts directory not found")

# Check TensorBoard logs
print("\n📊 TensorBoard Logs:")
if logs_dir.exists():
    log_files = list(logs_dir.glob('events.out.tfevents.*'))
    if log_files:
        print(f"  ✓ Found {len(log_files)} event files")
        print("  To view in Colab, run: %tensorboard --logdir model/logs")
    else:
        print("  ⚠ No event files found")
else:
    print(f"  ⚠ Logs directory not found")

print("\n" + "="*60)

## Step 9: View TensorBoard (Optional)

Visualize training metrics with TensorBoard.

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Display TensorBoard
log_dir = f"{PROJECT_PATH}/model/logs"
%tensorboard --logdir {log_dir}

## Step 10: Download Results to Local Machine

Download trained models and logs back to your computer.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import os

print("Preparing files for download...\n")

# Create a temporary directory for download
download_dir = '/tmp/breaking_fake_results'
os.makedirs(download_dir, exist_ok=True)

# Copy artifacts (models)
artifacts_src = f"{PROJECT_PATH}/model/artifacts"
artifacts_dst = f"{download_dir}/artifacts"
if os.path.exists(artifacts_src):
    shutil.copytree(artifacts_src, artifacts_dst, dirs_exist_ok=True)
    print(f"✓ Copied artifacts to {artifacts_dst}")

# Copy logs
logs_src = f"{PROJECT_PATH}/model/logs"
logs_dst = f"{download_dir}/logs"
if os.path.exists(logs_src):
    shutil.copytree(logs_src, logs_dst, dirs_exist_ok=True)
    print(f"✓ Copied logs to {logs_dst}")

print(f"\nDownloading files...\n")

# Download each file
for root, dirs, files_list in os.walk(download_dir):
    for file in files_list:
        file_path = os.path.join(root, file)
        print(f"  📥 {file}")

# Create a zip for easier download
zip_path = '/tmp/breaking_fake_results.zip'
shutil.make_archive('/tmp/breaking_fake_results', 'zip', download_dir)

print(f"\n✓ Created zip file: breaking_fake_results.zip")
print(f"\nDownloading...")
files.download(zip_path)

## Step 11: Test Inference (Optional)

Load the trained model and run inference on a test image.

In [ ]:
import torch
import timm
from pathlib import Path
from PIL import Image
import numpy as np

# Load best model
model_path = Path(PROJECT_PATH) / 'model' / 'artifacts' / 'breaking_fake_model_best.pth'

if model_path.exists():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load model
    model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=2)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    
    print(f"✓ Model loaded from {model_path.name}")
    print(f"  Epoch: {checkpoint.get('epoch')}")
    print(f"  Val Accuracy: {checkpoint.get('val_acc', 0):.4f}")
    print(f"  Best Val Accuracy: {checkpoint.get('best_val_acc', 0):.4f}")
else:
    print(f"❌ Model not found at {model_path}")

## Done! 🎉

Your Breaking-Fake model has been trained in Google Colab with:
- ✅ GPU acceleration (CUDA)
- ✅ Checkpoint management (resume/best model)
- ✅ TensorBoard logging
- ✅ Automated downloads

**Next steps:**
1. Download the `breaking_fake_results.zip` file
2. Extract and copy the models back to your local `model/artifacts/` folder
3. Run inference or continue fine-tuning locally